# Semantic Image Segmentation with AutoGluon

Semantic segmentation assigns a **class label to every pixel** in an image (unlike object detection which draws boxes). AutoGluon uses SegFormer and other transformer-based models via `MultiModalPredictor`.

**Task comparison:**
- Image Classification → one label per image
- Object Detection → bounding boxes with labels
- **Semantic Segmentation → per-pixel class label (a mask image)**

This notebook covers:
1. Data format: image paths + segmentation mask paths in a DataFrame
2. What a segmentation mask looks like (and common mistakes)
3. Training a segmentation model
4. Visualising predictions
5. Practical gotchas

In [ ]:
import autogluon
import torch
print('AutoGluon version:', autogluon.__version__)
print('CUDA available:', torch.cuda.is_available())

## 1. Understanding Segmentation Masks

A segmentation mask is a **grayscale PNG** (or single-channel image) where each pixel's value equals its class index:

```
Pixel value 0  →  background
Pixel value 1  →  cat
Pixel value 2  →  dog
```

> **Common mistake:** Some annotation tools save masks with value `255` for the background. AutoGluon expects `0` for background. Always check your mask values before training.

The mask must be the **same size** as its corresponding image.

In [ ]:
import os
import zipfile
import urllib.request

DATA_URL = 'https://autogluon.s3.amazonaws.com/datasets/camvid_tiny.zip'
DATA_DIR = './camvid_tiny'

if not os.path.exists(DATA_DIR):
    print('Downloading CamVid tiny dataset...')
    urllib.request.urlretrieve(DATA_URL, './camvid_tiny.zip')
    with zipfile.ZipFile('./camvid_tiny.zip', 'r') as z:
        z.extractall('.')
    os.remove('./camvid_tiny.zip')
    print('Done.')
else:
    print('Dataset already present.')

# Show directory structure
for root, dirs, files in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth < 2:
        for f in files[:3]:
            print(f'{indent}  {f}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Build DataFrames — one row per image, with matching mask path
def build_segmentation_df(image_dir: str, mask_dir: str) -> pd.DataFrame:
    rows = []
    for fname in sorted(os.listdir(image_dir)):
        if not fname.lower().endswith(('.jpg', '.png')):
            continue
        # Mask files often have a different suffix (e.g., _L.png for CamVid)
        mask_name = fname.replace('.jpg', '_L.png').replace('.png', '_L.png')
        mask_path = os.path.join(mask_dir, mask_name)
        if os.path.exists(mask_path):
            rows.append({
                'image': os.path.abspath(os.path.join(image_dir, fname)),
                'label': os.path.abspath(mask_path),
            })
    return pd.DataFrame(rows)

train_df = build_segmentation_df(
    os.path.join(DATA_DIR, 'train'),
    os.path.join(DATA_DIR, 'trainannot'),
)
val_df = build_segmentation_df(
    os.path.join(DATA_DIR, 'val'),
    os.path.join(DATA_DIR, 'valannot'),
)

print(f'Train: {len(train_df)} | Val: {len(val_df)}')
train_df.head()

In [ ]:
# Visualise an image and its segmentation mask side-by-side
row = train_df.iloc[0]
img  = np.array(Image.open(row['image']))
mask = np.array(Image.open(row['label']))

print(f'Image shape: {img.shape} | Mask shape: {mask.shape}')
print(f'Mask dtype: {mask.dtype} | Unique pixel values (classes): {np.unique(mask)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title('Image')
axes[0].axis('off')
axes[1].imshow(mask, cmap='tab20', interpolation='nearest')
axes[1].set_title('Segmentation Mask (pixel value = class index)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 2. Define the Category Mapping

AutoGluon needs to know which pixel value maps to which class name. Pass this via a `id2label` dictionary in hyperparameters.

In [ ]:
# CamVid class labels (abbreviated)
# Full list: https://mi.eng.cam.ac.uk/research/projects/VideoRec/CamVid/
CAMVID_CLASSES = {
    0:  'Sky',
    1:  'Building',
    2:  'Pole',
    3:  'Road',
    4:  'Pavement',
    5:  'Tree',
    6:  'SignSymbol',
    7:  'Fence',
    8:  'Car',
    9:  'Pedestrian',
    10: 'Bicyclist',
    11: 'Unlabelled',
}

num_classes = len(CAMVID_CLASSES)
print(f'Number of classes: {num_classes}')

## 3. Train the Segmentation Model

AutoGluon uses SegFormer (MiT-B0 to MiT-B5 encoders) by default — a lightweight but powerful transformer for dense prediction.

In [ ]:
from autogluon.multimodal import MultiModalPredictor

predictor = MultiModalPredictor(
    problem_type='semantic_segmentation',
    label='label',
    path='./ag_segmentation_model',
    eval_metric='iou',             # mean Intersection-over-Union
)

predictor.fit(
    train_data=train_df,
    tuning_data=val_df,            # optional: use val data for early stopping
    time_limit=180,
    hyperparameters={
        'model.sam.checkpoint_name': 'nvidia/segformer-b0-finetuned-ade-512-512',
        'env.num_workers': 2,
        'data.semantic_segmentation.num_classes': num_classes,
        'data.semantic_segmentation.id2label': CAMVID_CLASSES,
    },
)

## 4. Predict and Visualise

In [ ]:
# Predict returns a list of mask arrays (one per image)
predictions = predictor.predict(val_df.head(3))
print('Prediction type:', type(predictions[0]))
print('Predicted mask shape:', predictions[0].shape)

In [ ]:
# Overlay predicted mask on original image
idx = 0
img     = np.array(Image.open(val_df.iloc[idx]['image']))
gt_mask = np.array(Image.open(val_df.iloc[idx]['label']))
pr_mask = predictions[idx]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img); axes[0].set_title('Image'); axes[0].axis('off')
axes[1].imshow(gt_mask, cmap='tab20', interpolation='nearest')
axes[1].set_title('Ground Truth'); axes[1].axis('off')
axes[2].imshow(pr_mask, cmap='tab20', interpolation='nearest')
axes[2].set_title('Prediction'); axes[2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
metrics = predictor.evaluate(val_df)
print('Validation metrics:', metrics)

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **Background pixels as `255` instead of `0`** | All unlabelled pixels are treated as class 255 (invalid) → crashes or poor training | Re-map your masks: `mask[mask == 255] = 0` or recode your annotation pipeline |
| **Mask is RGB, not grayscale** | AutoGluon expects single-channel masks; RGB masks read as 3D arrays | Convert with `Image.open(mask_path).convert('L')` before checking pixel values |
| **Image and mask size mismatch** | Training fails or silently gives wrong results | Ensure every mask has exactly the same (H, W) as its image |
| **GPU is practically required** | SegFormer on CPU can take hours per epoch even on small datasets | Use CUDA; set `hyperparameters={'env.num_gpus': 1}` |
| **`id2label` not matching actual pixel values** | Class names are wrong in evaluation output | Inspect mask pixel values with `np.unique(mask)` first |
| **Mask filenames don't match image filenames** | Rows get mismatched images and masks | Build the DataFrame carefully; always validate by opening and comparing a few pairs |
| **`eval_metric='iou'` computes mean IoU** | Some classes with few examples drag down the mean | Check per-class IoU in the returned metrics dict |